# Simulation Walkthrough

This notebook steps through the full ABM simulation for a single 3-person
household from the real Walcheren sample data. Every LLM phase is its own
cell, so you can observe — and experiment with — the prompts and outputs
at each sub-step.

Compare this with `workflow/scripts/simulate.py`, which runs the same
logic in batch across all households using a thread pool.

**Requires** an `OPENAI_API_KEY` environment variable.

## Section 0 — Imports and helpers

In [ ]:
import math
import random
import re
from pathlib import Path

import pandas as pd
import yaml

from aibm import (
    Agent,
    Household,
    Skim,
    Zone,
    create_client,
    filter_pois,
    load_pois,
    load_prompt_config,
    load_skim,
)
from aibm.activity import Activity
from aibm.agent import ModeOption
from aibm.day_plan import DayPlan, compute_time_windows

DATA_DIR = Path("../data/processed")
CONFIG_PATH = Path("../workflow/config.yaml")
MODEL = "gpt-4o-mini"
N_ZONE_CANDIDATES = 12

In [2]:
# Helper functions copied verbatim from workflow/scripts/simulate.py

_ZONE_RE = re.compile(r"E(\d+)N(\d+)")


def _zones_from_specs(
    zone_specs: pd.DataFrame,
    all_pois: list,
) -> list[Zone]:
    """Build Zone objects from the zone_specs parquet.

    Uses the ``buurt_name`` column as the human-readable zone name when
    it is present and non-null; otherwise falls back to the grid code.
    """
    has_buurt = "buurt_name" in zone_specs.columns
    zones: list[Zone] = []
    for _, row in zone_specs.iterrows():
        zone_id = str(row["zone_id"])
        m = _ZONE_RE.match(zone_id)
        if m is None:
            continue
        x = int(m.group(1)) * 100 + 50
        y = int(m.group(2)) * 100 + 50
        if has_buurt and pd.notna(row["buurt_name"]):
            name = str(row["buurt_name"])
        else:
            name = zone_id
        zones.append(Zone(id=zone_id, name=name, x=float(x), y=float(y)))
    return zones


def _zone_poi_counts(pois: list, activity_types: set[str]) -> dict[str, int]:
    """Count POIs per zone for the given activity types."""
    counts: dict[str, int] = {}
    for p in pois:
        if p.activity_type in activity_types and p.zone_id:
            counts[p.zone_id] = counts.get(p.zone_id, 0) + 1
    return counts


def _reconstruct_household(hh_id: int, group: pd.DataFrame, model: str) -> Household:
    """Rebuild a Household and its Agents from a parquet row group."""
    first = group.iloc[0]
    hh = Household(
        id=str(hh_id),
        home_zone=str(first["home_zone"]),
        num_vehicles=int(first["num_vehicles"]),
        income_level=str(first["income_level"]),
    )
    for _, row in group.iterrows():
        agent = Agent(
            name=str(row["agent_name"]),
            age=int(row["age"]),
            employment=str(row["employment"]),
            has_license=bool(row["has_license"]),
            model=model,
        )
        hh.add_member(agent)
    return hh


def _sample_zones(
    home_zone: str | None,
    all_zones: list[Zone],
    skims: list[Skim],
    n: int,
    poi_counts: dict[str, int] | None = None,
) -> tuple[list[Zone], dict[str, dict[str, float]]]:
    """Return a random sample of reachable zones and their travel times.

    When *poi_counts* is provided, only zones that contain at least one
    relevant POI are considered. Each returned zone's ``poi_count``
    attribute is stamped with the count from *poi_counts*.

    Candidates are drawn uniformly at random from all reachable eligible
    zones, so the LLM is not mechanically biased toward nearby destinations.
    """
    if home_zone is None:
        return [], {}

    eligible = (
        [z for z in all_zones if poi_counts.get(z.id, 0) > 0]
        if poi_counts is not None
        else all_zones
    )

    reachable: list[Zone] = []
    for z in eligible:
        min_tt = min(
            (sk.travel_time(home_zone, z.id) for sk in skims),
            default=math.inf,
        )
        if min_tt < math.inf:
            reachable.append(z)

    candidates = random.sample(reachable, min(n, len(reachable)))

    if poi_counts is not None:
        for z in candidates:
            z.poi_count = poi_counts.get(z.id, 0)

    travel_times: dict[str, dict[str, float]] = {}
    for z in candidates:
        tt_by_mode: dict[str, float] = {}
        for sk in skims:
            tt = sk.travel_time(home_zone, z.id)
            if tt < math.inf:
                tt_by_mode[sk.mode] = tt
        travel_times[z.id] = tt_by_mode

    return candidates, travel_times


def _build_mode_options(
    origin: str,
    destination: str,
    skims: list[Skim],
    has_vehicle: bool,
) -> list[ModeOption]:
    """Build available ModeOptions for an OD pair.

    Excludes car when has_vehicle is False.
    """
    options: list[ModeOption] = []
    for sk in skims:
        tt = sk.travel_time(origin, destination)
        if tt >= math.inf:
            continue
        if sk.mode == "car" and not has_vehicle:
            continue
        options.append(ModeOption(mode=sk.mode, travel_time=tt))
    return options

In [3]:
def fmt_time(mins: float | None) -> str:
    """Convert minutes-from-midnight to HH:MM string."""
    if mins is None:
        return "--:--"
    h = int(mins) // 60
    m = int(mins) % 60
    return f"{h:02d}:{m:02d}"


print("fmt_time(480) =", fmt_time(480), "  (should be 08:00)")
print("fmt_time(1020) =", fmt_time(1020), "  (should be 17:00)")

fmt_time(480) = 08:00   (should be 08:00)
fmt_time(1020) = 17:00   (should be 17:00)


## Section 1 — Load data

Four data sources drive the simulation:

- **sample** — the synthetic population, one row per agent
- **zone_specs** — grid-cell attributes used to build `Zone` objects
- **pois** — Points of Interest (shops, restaurants, etc.) with zone IDs
- **skims** — travel-time matrices, one per mode (car, bike, walk, transit)

In [4]:
sample = pd.read_parquet(DATA_DIR / "walcheren_sample.parquet")
zone_specs_df = pd.read_parquet(DATA_DIR / "walcheren_zone_specs.parquet")
all_pois = load_pois(DATA_DIR / "walcheren_pois.parquet")
skims = [
    load_skim(DATA_DIR / "walcheren_skim_car.omx", "car"),
    load_skim(DATA_DIR / "walcheren_skim_bike.omx", "bike"),
    load_skim(DATA_DIR / "walcheren_skim_walk.omx", "walk"),
    load_skim(DATA_DIR / "walcheren_skim_transit.omx", "transit"),
]
all_zones = _zones_from_specs(zone_specs_df, all_pois)
work_counts = _zone_poi_counts(all_pois, {"work"})
school_counts = _zone_poi_counts(all_pois, {"school", "escort"})

print(f"Agents in sample : {len(sample)}")
print(f"Zones            : {len(all_zones)}")
print(f"POIs             : {len(all_pois)}")
print(f"Skim modes       : {[sk.mode for sk in skims]}")
print(f"Work zones       : {len(work_counts)}")
print(f"School zones     : {len(school_counts)}")

/home/martijn/dev/aibm/.venv/lib/python3.12/site-packages/tables/attributeset.py:322: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=()`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  value = self._g_getattr(self._v_node, name)


Agents in sample : 25
Zones            : 2670
POIs             : 5919
Skim modes       : ['car', 'bike', 'walk', 'transit']
Work zones       : 557
School zones     : 69


## Section 2 — Select the household

We pick the first 3-person household in the sample. Using real data means
every zone ID, travel time, and POI is grounded in the actual study area.

In [5]:
sizes = sample.groupby("household_id").size()
hh_ids_3 = sizes[sizes == 3].index.tolist()
print(f"3-person households in sample: {len(hh_ids_3)}")

TARGET_HH_ID = hh_ids_3[0]
group = sample[sample["household_id"] == TARGET_HH_ID]
print(f"\nSelected household_id: {TARGET_HH_ID}")
print()
print(
    group[
        [
            "agent_name",
            "age",
            "employment",
            "has_license",
            "home_zone",
            "num_vehicles",
            "income_level",
        ]
    ].to_string(index=False)
)

3-person households in sample: 1

Selected household_id: 48265

 agent_name  age employment  has_license  home_zone  num_vehicles income_level
Agent 99903   33   employed         True E0276N3869             0          low
Agent 99904   37   employed        False E0276N3869             0          low
Agent 99905   52    student         True E0276N3869             0          low


In [6]:
client = create_client(MODEL)
hh = _reconstruct_household(TARGET_HH_ID, group, MODEL)

print(f"Household  id={hh.id}")
print(f"  home_zone    : {hh.home_zone}")
print(f"  num_vehicles : {hh.num_vehicles}")
print(f"  income_level : {hh.income_level}")
print()
for a in hh.members:
    lic = "licence" if a.has_license else "no licence"
    print(f"  {a.name}  age={a.age}  {a.employment}  {lic}")

Household  id=48265
  home_zone    : E0276N3869
  num_vehicles : 0
  income_level : low

  Agent 99903  age=33  employed  licence
  Agent 99904  age=37  employed  no licence
  Agent 99905  age=52  student  licence


In [ ]:
# Load prompt overrides from workflow/config.yaml (simulation.prompts section).
# Any step not listed there keeps its built-in default.
with open(CONFIG_PATH) as f:
    _cfg = yaml.safe_load(f)

pc = load_prompt_config(_cfg.get("simulation", {}).get("prompts", {}))
print("Prompt config loaded.")

## Section 3 — Per-agent planning

Each sub-section runs one LLM phase for every household member in turn.
Results are printed first; the raw prompt is printed below so you can see
exactly what the model received.

### Step 3.1 — Persona generation

The persona is a 1–2 sentence behavioural profile that captures how this
person typically travels (preferred modes, time constraints, lifestyle).
It is stored on `agent.persona` and included in every subsequent prompt
to give the LLM consistent character context.

In [ ]:
for agent in hh.members:
    persona, prompt = agent.generate_persona(
        client=client, household=hh, step=pc.persona
    )
    print(f"=== {agent.name} ===")
    print(f"Persona: {persona}")
    print(f"\nPrompt:\n{prompt}")
    print("=" * 60, "\n")

### Step 3.2 — Long-term zone choice

Employed agents choose a work zone; students choose a school zone.
`_sample_zones` draws `N_ZONE_CANDIDATES` zones at random from all
reachable zones that contain at least one relevant POI, then passes
that sample to the LLM. Agents with other employment statuses skip
this step.

In [ ]:
for agent in hh.members:
    if agent.employment not in ("employed", "student"):
        print(f"{agent.name}: {agent.employment} — no zone choice\n")
        continue

    if agent.employment == "employed":
        candidates, travel_times = _sample_zones(
            agent.home_zone, all_zones, skims, N_ZONE_CANDIDATES, work_counts
        )
        zone_id, reasoning, prompt = agent.choose_work_zone(
            candidates, travel_times, client=client, step=pc.zone_choice
        )
        label = "work zone"
    else:
        candidates, travel_times = _sample_zones(
            agent.home_zone, all_zones, skims, N_ZONE_CANDIDATES, school_counts
        )
        zone_id, reasoning, prompt = agent.choose_school_zone(
            candidates, travel_times, client=client, step=pc.zone_choice
        )
        label = "school zone"

    print(f"=== {agent.name} — chosen {label}: {zone_id} ===")
    print(f"Reasoning: {reasoning}")
    print(f"\nPrompt:\n{prompt}")
    print("=" * 60, "\n")

### Aside — Custom prompt overrides

The `PromptConfig` system lets you override any section of any prompt
without touching source code. Here's a quick demo building a custom
`mode_choice` prompt section that emphasises sustainability:

In [ ]:
# Example: override the mode_choice role to emphasise sustainability
from aibm.prompts import build_prompt
from aibm.prompts import load_prompt_config as _lpc

custom_cfg = {
    "mode_choice": {
        "role": (
            "You are {agent_name}, an environmentally conscious "
            "traveller deciding how to get around."
        ),
    },
}
custom_pc = _lpc(custom_cfg)

# Preview what the assembled prompt looks like (no LLM call)
preview = build_prompt(
    custom_pc.mode_choice,
    {"agent_name": hh.members[0].name},
    "Background:\nAge: 33\n\nAvailable options:\n- bike: 12 minutes\n- car: 5 minutes",
)
print(preview)

### Step 3.3 — Activity generation

The LLM produces a list of out-of-home activities for the day.
Mandatory activities (work, school) are tagged `is_flexible=False`;
discretionary ones (shopping, leisure, eating_out, …) may be flexible.

In [ ]:
agent_activities: dict[str, list[Activity]] = {}

for agent in hh.members:
    activities, prompt = agent.generate_activities(client=client, step=pc.activities)
    agent_activities[agent.id] = activities

    mandatory = [a.type for a in activities if not a.is_flexible]
    discretionary = [a.type for a in activities if a.is_flexible]

    print(f"=== {agent.name} ===")
    print(f"Mandatory    : {mandatory}")
    print(f"Discretionary: {discretionary}")
    print(f"\nPrompt:\n{prompt}")
    print("=" * 60, "\n")

### Step 3.4 — Schedule mandatory activities

The LLM assigns start/end times (minutes from midnight) to each mandatory
activity. Travel times between consecutive activities are included in the
prompt so the model can leave realistic gaps.

In [ ]:
agent_mandatory_plans: dict[str, DayPlan] = {}

for agent in hh.members:
    mandatory = [a for a in agent_activities[agent.id] if not a.is_flexible]
    plan, prompt = agent.schedule_activities(
        mandatory, client=client, skims=skims, step=pc.scheduling
    )
    agent_mandatory_plans[agent.id] = plan

    print(f"=== {agent.name} ===")
    for act in plan.activities:
        print(
            f"  {act.type:20s}  {fmt_time(act.start_time)} – {fmt_time(act.end_time)}"
        )
    print(f"\nPrompt:\n{prompt}")
    print("=" * 60, "\n")

In [ ]:
agent_time_windows: dict[str, list] = {}

for agent in hh.members:
    mandatory_plan = agent_mandatory_plans[agent.id]
    windows = compute_time_windows(mandatory_plan, skims, home_zone=agent.home_zone)
    agent_time_windows[agent.id] = windows

    print(f"=== {agent.name} ===")
    if not windows:
        print("  (no free windows — mandatory activities fill the whole day)")
    for w in windows:
        print(
            f"  {w.start / 60:.1f}h – {w.end / 60:.1f}h "
            f"({w.duration:.0f} min) | "
            f"from {w.preceding_location} → {w.following_location}"
        )
    print()

### Step 3.5 — Plan discretionary activities

A single LLM call assigns both destination (from real POIs in the study
area) and a time slot to all discretionary activities in the context of
the mandatory anchors already fixed above.

Activities with no matching POIs are silently dropped, so the final day
plan may have fewer discretionary activities than the list from step 3.3.

In [ ]:
agent_day_plans: dict[str, DayPlan] = {}

for agent in hh.members:
    activities = agent_activities[agent.id]
    mandatory_plan = agent_mandatory_plans[agent.id]
    discretionary = [a for a in activities if a.is_flexible]

    # Find POIs for each discretionary activity type
    pois_by_type: dict[str, list] = {}
    for act in discretionary:
        if act.type not in pois_by_type:
            pts = filter_pois(all_pois, act.type)
            if pts:
                pois_by_type[act.type] = pts

    # Only plan discretionary activities that have POI candidates
    disc_with_pois = [a for a in discretionary if a.type in pois_by_type]

    planned_disc: list[Activity] = []
    prompt_disc: str | None = None
    if disc_with_pois:
        time_windows = agent_time_windows[agent.id]
        planned_disc, prompt_disc = agent.plan_discretionary_activities(
            mandatory_plan.activities,
            disc_with_pois,
            pois_by_type,
            skims,
            client=client,
            time_windows=time_windows,
            step=pc.discretionary,
        )

    # Combine mandatory + discretionary; keep only activities with a location
    routable = [
        a for a in mandatory_plan.activities + planned_disc if a.location is not None
    ]
    day_plan = DayPlan(activities=sorted(routable, key=lambda a: a.start_time or 0))
    agent_day_plans[agent.id] = day_plan

    print(f"=== {agent.name} — full day plan ===")
    if not day_plan.activities:
        print("  (no routable activities)")
    for act in day_plan.activities:
        flex = "flex" if act.is_flexible else "fixed"
        print(
            f"  {act.type:20s}  "
            f"{fmt_time(act.start_time)} – {fmt_time(act.end_time)}  "
            f"[{flex}]  loc={act.location}"
        )
    if prompt_disc:
        print(f"\nDiscretionary prompt:\n{prompt_disc}")
    else:
        print("\n(no discretionary LLM call — no POI candidates available)")
    print("=" * 60, "\n")

### Step 3.6 — Validate day plans

`DayPlan.validate()` checks for:
- times outside the 0–1440 minute window
- overlapping activities
- work durations outside the 4–10 hour range

An empty list means the plan looks valid.

In [ ]:
for agent in hh.members:
    warnings = agent_day_plans[agent.id].validate()
    status = "OK" if not warnings else f"{len(warnings)} warning(s)"
    print(f"{agent.name}: {status}")
    for w in warnings:
        print(f"  WARNING: {w}")

### Step 3.7 — Build tours

Pure logic — no LLM call. `build_tours` groups the day's trips into
home-based chains: each time the agent leaves home a new tour starts,
and it closes when they return. A tour that never returns home stays open.

In [ ]:
for agent in hh.members:
    day_plan = agent_day_plans[agent.id]
    if not day_plan.activities:
        print(f"{agent.name}: no activities — skipping tour build\n")
        continue
    agent.build_tours(day_plan, skims=skims)
    print(f"=== {agent.name}: {len(day_plan.tours)} tour(s) ===")
    for i, tour in enumerate(day_plan.tours):
        closed = "closed" if tour.is_closed else "open"
        print(f"  Tour {i} ({closed}):")
        for trip in tour.trips:
            print(
                f"    {trip.origin} -> {trip.destination}  "
                f"depart={fmt_time(trip.departure_time)}"
            )
    print()

## Section 4 — Household coordination

After each member has their own day plan, three household-level decisions
are made: shared activities, escorting children, and vehicle allocation.

### Phase 1a — Joint activities

The LLM proposes 0–2 shared discretionary activities that fit everyone's
schedule. For each proposal, participating members have the activity
injected into their day plan and their tours rebuilt.

In [ ]:
# Build per-member fixed (non-flexible) activity schedules
member_schedules = {
    agent.id: [a for a in agent_day_plans[agent.id].activities if not a.is_flexible]
    for agent in hh.members
    if agent_day_plans.get(agent.id) is not None
}

# POI candidates for joint discretionary activity types
joint_pois_by_type = {
    t: filter_pois(all_pois, t)
    for t in ("shopping", "leisure", "eating_out")
    if filter_pois(all_pois, t)
}

joint_activities, prompt_joint = hh.plan_joint_activities(
    member_schedules,
    joint_pois_by_type,
    skims,
    client=client,
    model=MODEL,
    step=pc.joint_activities,
)

print(f"Joint activities proposed: {len(joint_activities)}\n")
for ja in joint_activities:
    member_names = [a.name for a in hh.members if a.id in ja.participant_ids]
    print(
        f"  {ja.activity.type}  "
        f"{fmt_time(ja.activity.start_time)} – {fmt_time(ja.activity.end_time)}  "
        f"loc={ja.activity.location}"
    )
    print(f"  Participants : {', '.join(member_names)}")
    print(f"  Reasoning    : {ja.reasoning}")
    print()

print(f"\nPrompt:\n{prompt_joint}")

In [ ]:
# Inject joint activities into participating members' day plans and rebuild tours
for ja in joint_activities:
    for agent in hh.members:
        if agent.id in ja.participant_ids:
            dp = agent_day_plans[agent.id]
            dp.inject_joint(ja.activity)
            dp.tours = []
            agent.build_tours(dp, skims=skims)
            print(
                f"{agent.name}: injected '{ja.activity.type}' — "
                f"now {len(dp.activities)} activities, "
                f"{len(dp.tours)} tour(s)"
            )

### Phase 1b — Escort check

Children under 12 cannot travel alone. `members_needing_escort()` checks
the household's members and returns any children below the age threshold.
`potential_escorts` lists adults (18+) with a driving licence.

If children are found, `plan_escort_trips` calls the LLM to assign which
parent handles drop-off and which handles pick-up, then injects escort
activities into the parent's day plan.

In [ ]:
children = hh.members_needing_escort()
escorts = hh.potential_escorts

print(f"Children needing escort : {[a.name for a in children]}")
print(f"Potential escorts       : {[a.name for a in escorts]}")

if children and escorts:
    child_activities = {}
    for agent in hh.members:
        if agent in children:
            dp = agent_day_plans.get(agent.id)
            if dp is not None:
                school_acts = [
                    a
                    for a in dp.activities
                    if a.type == "school" and a.location is not None
                ]
                if school_acts:
                    child_activities[agent.id] = school_acts

    parent_plans = {
        agent.id: agent_day_plans[agent.id]
        for agent in escorts
        if agent.id in agent_day_plans
    }

    if child_activities and parent_plans:
        updated_plans, prompt_escort = hh.plan_escort_trips(
            child_activities,
            parent_plans,
            skims,
            client=client,
            model=MODEL,
            step=pc.escort,
        )
        for aid, dp in updated_plans.items():
            agent_day_plans[aid] = dp
        print(f"\nEscort prompt:\n{prompt_escort}")
    else:
        print("\nNo escort trips needed (no school activities with locations).")
else:
    print("\nNo children under 12 in this household — escort phase skipped.")

### Phase 2 — Vehicle allocation

The LLM (or a fast path) decides which household member gets the car for
each tour.

Fast paths that skip the LLM:
- `num_vehicles == 0` → nobody gets a car
- `num_vehicles >= licensed_adults_with_tours` → everyone gets a car

When the LLM is called the result is a `dict[agent_id, list[bool]]` — one
boolean per tour indicating vehicle access.

In [ ]:
member_tours = {
    agent.id: agent_day_plans[agent.id].tours
    for agent in hh.members
    if agent_day_plans.get(agent.id) and agent_day_plans[agent.id].tours
}

vehicle_alloc: dict[str, list[bool]] = {}
prompt_veh = ""

if member_tours:
    vehicle_alloc, prompt_veh = hh.allocate_vehicles(
        member_tours,
        skims,
        client=client,
        model=MODEL,
        step=pc.vehicle_allocation,
    )

print("Vehicle allocation:")
for agent in hh.members:
    access = vehicle_alloc.get(agent.id, [])
    print(f"  {agent.name}: {access}")

if prompt_veh:
    print(f"\nPrompt:\n{prompt_veh}")
else:
    n_licensed = sum(
        1 for a in hh.members if a.has_license and a.age >= 18 and a.id in member_tours
    )
    print(
        f"\n(No LLM call — fast path taken. "
        f"num_vehicles={hh.num_vehicles}, "
        f"licensed adults with tours={n_licensed})"
    )

## Section 5 — Mode choice

For each tour the available modes are built from the skim matrices. Car
is excluded when the agent has no vehicle access for that tour.
The LLM picks one mode and applies it to every trip in the tour.

In [ ]:
for agent in hh.members:
    day_plan = agent_day_plans.get(agent.id)
    if not day_plan or not day_plan.tours:
        print(f"{agent.name}: no tours\n")
        continue

    access = vehicle_alloc.get(agent.id)
    print(f"=== {agent.name} ===")

    for tour_idx, tour in enumerate(day_plan.tours):
        if not tour.trips:
            continue
        first = tour.trips[0]
        has_car = access[tour_idx] if access and tour_idx < len(access) else False
        options = _build_mode_options(first.origin, first.destination, skims, has_car)

        if not options:
            print(f"  Tour {tour_idx}: no mode options available")
            continue

        mc, prompt_mode = agent.choose_tour_mode(
            tour, options, client=client, household=hh, step=pc.mode_choice
        )

        od = f"{first.origin} -> {tour.trips[-1].destination}"
        modes_avail = [f"{o.mode} {o.travel_time:.0f}min" for o in options]
        print(f"  Tour {tour_idx}: {od}")
        print(f"  Available modes : {', '.join(modes_avail)}")
        print(f"  Chosen mode     : {mc.option.mode}")
        print(f"  Reasoning       : {mc.reasoning}")
        print(f"\n  Prompt:\n{prompt_mode}")
        print()
    print("=" * 60, "\n")

## Section 6 — Final summary

A complete household view: one table per agent and an overall mode share.

In [ ]:
from collections import Counter

all_modes: list[str] = []

for agent in hh.members:
    day_plan = agent_day_plans.get(agent.id)
    print(f"{'=' * 60}")
    print(f"Agent : {agent.name}  (age={agent.age}, {agent.employment})")
    print(f"Persona : {agent.persona}")

    if agent.work_zone:
        print(f"Work zone   : {agent.work_zone}")
    if agent.school_zone:
        print(f"School zone : {agent.school_zone}")

    if not day_plan or not day_plan.activities:
        print("  (no day plan)")
        print()
        continue

    print("\nActivities:")
    print(f"  {'Type':<20}  {'Start':>5}  {'End':>5}  {'Flexible':>8}  Location")
    print(f"  {'-' * 20}  {'-' * 5}  {'-' * 5}  {'-' * 8}  {'-' * 20}")
    for act in day_plan.activities:
        flex = "yes" if act.is_flexible else "no"
        print(
            f"  {act.type:<20}  "
            f"{fmt_time(act.start_time):>5}  "
            f"{fmt_time(act.end_time):>5}  "
            f"{flex:>8}  "
            f"{act.location or ''}"
        )

    if day_plan.tours:
        print("\nTours:")
        for i, tour in enumerate(day_plan.tours):
            closed = "closed" if tour.is_closed else "open"
            mode = tour.trips[0].mode if tour.trips else "?"
            print(f"  Tour {i} ({closed}, mode={mode}):")
            for trip in tour.trips:
                print(
                    f"    {trip.origin} -> {trip.destination}  "
                    f"depart={fmt_time(trip.departure_time)}  "
                    f"mode={trip.mode or '?'}"
                )
            all_modes.extend(trip.mode for trip in tour.trips if trip.mode is not None)
    print()

# Mode share
print("=" * 60)
print("Household mode share")
print("=" * 60)
total_trips = len(all_modes)
if total_trips == 0:
    print("  (no trips with mode assigned)")
else:
    for mode, count in Counter(all_modes).most_common():
        pct = count / total_trips * 100
        print(f"  {mode:<10} {count:3d} trips  ({pct:.0f}%)")
    print(f"  {'Total':<10} {total_trips:3d} trips")